# Modelado — Colono IA

Regresión lineal que predice los puntos de una pareja de vértices, y agrupamiento que descubre
las familias de estrategia.

Continúa de `01_eda.ipynb`, que decidió qué variables tienen sentido y cuáles son redundantes.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, silhouette_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RUTA = "../data/processed/parejas.csv"   # en Colab: "data/processed/parejas.csv"
OBJETIVO = "puntos"

datos = pd.read_csv(RUTA)
print(f"{len(datos):,} parejas de {datos['tablero_id'].nunique()} tableros")

## 1. Qué variables entran al modelo

El EDA mostró que muchas variables miden lo mismo. Aquí se usa un conjunto **compacto de 12**,
elegido por dos criterios: que aporte información y que **se pueda explicar en voz alta**, porque
los coeficientes son la explicación que la app le da al usuario.

Más abajo se compara contra el modelo con todas las variables para comprobar que no se pierde
capacidad predictiva.

In [ ]:
VARIABLES = [
    "produccion_efectiva",    # producción ajustada por lo que se puede gastar
    "par_camino",             # madera con ladrillo
    "par_ciudad",             # trigo con mineral, ponderada por su costo
    "trio_desarrollo",        # trigo, oveja y mineral
    "cuarteto_poblado",       # los cuatro recursos de un poblado
    "desequilibrio",          # concentración de la producción
    "puerto_alineado",        # puerto 2:1 del recurso que más produces
    "puerto_generico",        # puerto 3:1
    "vertices_expansion_2",   # espacio para crecer
    "jugadores",              # 3 o 4
    "num_hexagonos",          # distingue costa de centro
    "solapamiento",           # si los dos poblados comparten hexágono
]

TODAS = [c for c in datos.columns if c not in ["tablero_id", "pareja_id", OBJETIVO]]
print(f"Compacto: {len(VARIABLES)} variables | Completo: {len(TODAS)}")

## 2. Partición por tablero

Un tablero entero va a entrenamiento o a prueba, nunca partido. Es como se usa el modelo de
verdad: la app recibe tableros que nunca ha visto.

In [ ]:
particion = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
i_entrena, i_prueba = next(particion.split(datos, datos[OBJETIVO], groups=datos["tablero_id"]))
entrena, prueba = datos.iloc[i_entrena], datos.iloc[i_prueba]

print(f"Entrenamiento: {len(entrena):,} parejas de {entrena['tablero_id'].nunique()} tableros")
print(f"Prueba:        {len(prueba):,} parejas de {prueba['tablero_id'].nunique()} tableros")
print(f"Tableros compartidos: {len(set(entrena['tablero_id']) & set(prueba['tablero_id']))}")

## 3. La regresión

Se estandarizan las variables antes de ajustar. Así los coeficientes quedan en la misma escala y
se pueden comparar entre sí: cada uno dice cuántos puntos cambia la predicción al mover esa
variable una desviación estándar.

In [ ]:
modelo = make_pipeline(StandardScaler(), LinearRegression())
modelo.fit(entrena[VARIABLES], entrena[OBJETIVO])
prediccion = modelo.predict(prueba[VARIABLES])

r2 = r2_score(prueba[OBJETIVO], prediccion)
mae = mean_absolute_error(prueba[OBJETIVO], prediccion)

print(f"R²  = {r2:.3f}")
print(f"MAE = {mae:.3f} puntos")
print(f"Desviación del objetivo = {prueba[OBJETIVO].std():.3f} puntos")
print(f"\nEl error medio es {mae / prueba[OBJETIVO].std():.0%} de la variación natural del objetivo.")

Un R² de aproximadamente 0.63 significa que el modelo explica cerca de dos tercios de la variación
en los puntos. El tercio restante es en buena medida **irreducible**: los dados son aleatorios, y
dos partidas idénticas desde la misma pareja dan resultados distintos. La variable objetivo ya es
un promedio de 40 partidas, pero conserva ruido de alrededor de 0.11 puntos.

## 4. Comparación con el modelo completo

In [ ]:
completo = make_pipeline(StandardScaler(), LinearRegression())
completo.fit(entrena[TODAS], entrena[OBJETIVO])
pred_completo = completo.predict(prueba[TODAS])

comparacion = pd.DataFrame({
    "variables": [len(VARIABLES), len(TODAS)],
    "R²": [r2, r2_score(prueba[OBJETIVO], pred_completo)],
    "MAE": [mae, mean_absolute_error(prueba[OBJETIVO], pred_completo)],
}, index=["Compacto", "Completo"]).round(3)
print(comparacion.to_string())

El modelo completo apenas gana unas milésimas de R² a cambio de casi triplicar el número de
variables, y con ellas aparecen coeficientes de signo contrario al esperado, efecto de las
variables que se solapan. **Se elige el compacto.**

Esto es exactamente lo que se busca en este proyecto: si hiciera falta un modelo de ensamble para
predecir bien, sería señal de que la ingeniería de variables no fue suficientemente buena. Aquí una
regresión lineal basta, y a cambio se conserva la interpretabilidad.

## 5. Qué dicen los coeficientes

In [ ]:
coeficientes = (
    pd.Series(modelo[-1].coef_, index=VARIABLES)
    .sort_values(key=abs, ascending=False)
)

fig = px.bar(
    coeficientes.iloc[::-1], orientation="h",
    title="Coeficientes estandarizados de la regresión",
    labels={"value": "Efecto en puntos por desviación estándar", "index": ""},
    color=coeficientes.iloc[::-1].values,
    color_continuous_scale="RdBu", range_color=[-0.6, 0.6],
)
fig.update_layout(height=520, coloraxis_showscale=False)
fig.show()

coeficientes.round(3).to_frame("coeficiente")

### El puerto: la correlación decía una cosa y la regresión dice otra

En el EDA, `puerto_alineado` tenía una correlación **negativa** con los puntos. En la regresión su
coeficiente sale **positivo**.

No es una contradicción, es una variable de confusión. Los puertos están en la costa, y los
vértices costeros tocan uno o dos hexágonos en vez de tres, así que producen menos. La correlación
simple mezcla dos efectos opuestos: tener puerto ayuda, estar en la costa perjudica.

La regresión los separa porque incluye la producción al mismo tiempo. Leído correctamente, el
coeficiente dice: **a igualdad de producción, tener el puerto 2:1 del recurso que más produces
suma puntos.** Esa es la jugada de concentrar un recurso y convertirlo a mitad de precio.

Es un buen recordatorio de por qué una correlación no basta para concluir.

In [ ]:
# El mismo efecto, visto directamente: parejas de producción parecida, con y sin puerto alineado
banda = datos[datos["produccion_efectiva"].between(12, 16)]
print("Parejas con producción efectiva entre 12 y 16:\n")
print(
    banda.groupby(banda["puerto_alineado"] > 0)[OBJETIVO]
    .agg(["count", "mean"]).round(2)
    .rename(index={False: "sin puerto alineado", True: "con puerto alineado"})
    .to_string()
)

## 6. La métrica que de verdad importa

El R² mide qué tan bien predice el modelo. Pero la app no predice: **elige**. Lo que importa es si
la pareja que recomienda es mejor que la que elegiría el usuario por su cuenta.

Se compara, en cada tablero de prueba:

- **Modelo**: la pareja con mayor predicción
- **Pips**: la pareja con más pips, que es lo que hace cualquier jugador
- **Tope**: la mejor pareja real del tablero, el máximo alcanzable
- **Azar**: el promedio de todas las parejas candidatas

In [ ]:
evaluacion = prueba.copy()
evaluacion["prediccion"] = prediccion

filas = []
for _, tablero in evaluacion.groupby("tablero_id"):
    filas.append({
        "modelo": tablero.loc[tablero["prediccion"].idxmax(), OBJETIVO],
        "pips":   tablero.loc[tablero["pips_totales"].idxmax(), OBJETIVO],
        "tope":   tablero[OBJETIVO].max(),
        "azar":   tablero[OBJETIVO].mean(),
    })
resultado = pd.DataFrame(filas).mean().round(2)
print(resultado.to_string())

margen = resultado["tope"] - resultado["azar"]
print(f"\nDel margen disponible sobre elegir al azar:")
print(f"  el modelo captura {(resultado['modelo'] - resultado['azar']) / margen:.0%}")
print(f"  elegir por pips captura {(resultado['pips'] - resultado['azar']) / margen:.0%}")
print(f"\nVentaja del modelo sobre los pips: {resultado['modelo'] - resultado['pips']:+.2f} puntos")

**Este es el número del proyecto.** Elegir con el modelo vale alrededor de un punto de victoria más
que elegir por pips, que es el criterio que usa todo jugador en la mesa. En una partida que se gana
con 10 puntos, un punto es mucho.

## 7. Modelación no supervisada: descubrir las estrategias

Hasta aquí las familias de estrategia estarían definidas a mano. Con agrupamiento se dejan salir de
los datos.

El agrupamiento **no se hace sobre el reparto crudo de recursos**, porque eso solo separa "qué
recurso tienes más", que no es una estrategia: al probarlo, los grupos resultantes sacaban
prácticamente los mismos puntos. Se agrupa sobre lo que la pareja **puede construir**, que es lo que
define cómo se juega.

In [ ]:
CAPACIDADES = ["par_camino", "par_ciudad", "trio_desarrollo",
               "cuarteto_poblado", "puerto_alineado", "desequilibrio"]

escalador = StandardScaler().fit(datos[CAPACIDADES])
Z = escalador.transform(datos[CAPACIDADES])

for k in range(2, 7):
    etiquetas = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Z)
    s = silhouette_score(Z, etiquetas, sample_size=3000, random_state=0)
    print(f"k = {k}  silueta = {s:.3f}")

La silueta sigue subiendo con más grupos, lo que es habitual y no decide por sí solo. Se eligen
**cuatro** porque es donde los grupos se corresponden con estrategias reconocibles del juego y con
resultados claramente distintos, que es el criterio útil para el producto.

In [ ]:
agrupamiento = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Z)
datos["grupo"] = agrupamiento.labels_

perfil = datos.groupby("grupo").agg(
    parejas=(OBJETIVO, "size"),
    puntos=(OBJETIVO, "mean"),
    camino=("par_camino", "mean"),
    ciudad=("par_ciudad", "mean"),
    desarrollo=("trio_desarrollo", "mean"),
    poblado=("cuarteto_poblado", "mean"),
    puerto=("puerto_alineado", "mean"),
    desequilibrio=("desequilibrio", "mean"),
    pips=("pips_totales", "mean"),
).round(2).sort_values("puntos", ascending=False)
perfil

### Las cuatro familias, con el nombre que les corresponde

El algoritmo agrupa; ponerles nombre es trabajo de quien conoce el juego. Mirando los perfiles:

**Expansión.** Mucho `par_camino` y `cuarteto_poblado`. Construye caminos y poblados desde temprano
y pelea la carta de camino más largo. Es la familia con más puntos.

**Ciudades y desarrollo.** Alto en `par_ciudad` y `trio_desarrollo`. Crece hacia arriba subiendo
poblados a ciudades y comprando cartas. Segunda en puntos.

**Puerto.** Se distingue por `puerto_alineado`. Es la de **menos producción total** y aun así no es
la peor: concentra un recurso y lo convierte a mitad de precio. Es una jugada de nicho que
funciona, y confirma con datos una intuición de jugadora.

**Desequilibrada.** Todo bajo y el `desequilibrio` más alto. Produce sin poder gastar. Es la peor, y
es justo la que un jugador elige cuando se guía solo por los pips.

In [ ]:
fig = px.scatter(
    datos.sample(min(4000, len(datos)), random_state=0),
    x="pips_totales", y=OBJETIVO, color=datos["grupo"].astype(str),
    opacity=0.5,
    title="Las cuatro familias de estrategia",
    labels={"pips_totales": "Pips de la pareja", OBJETIVO: "Puntos", "color": "Grupo"},
)
fig.update_layout(height=520)
fig.show()

## 8. Guardar el modelo para la aplicación

In [ ]:
final = make_pipeline(StandardScaler(), LinearRegression()).fit(datos[VARIABLES], datos[OBJETIVO])
agrupamiento_final = KMeans(n_clusters=4, n_init=10, random_state=0).fit(Z)

joblib.dump({
    "modelo": final,
    "variables": VARIABLES,
    "agrupamiento": agrupamiento_final,
    "escalador_capacidades": escalador,
    "capacidades": CAPACIDADES,
    "r2_prueba": round(r2, 3),
    "mae_prueba": round(mae, 3),
}, "../modelos/colono.joblib")

print("Guardado en modelos/colono.joblib")

## 9. Conclusiones

**Una regresión lineal con 12 variables basta.** R² de aproximadamente 0.63 y un error medio de
0.65 puntos, frente a una variación natural de 1.5. Añadir 20 variables más sube el R² unas
milésimas y estropea la interpretación.

**El modelo elige mejor que los pips.** Alrededor de un punto de victoria más por partida frente al
criterio que usa cualquier jugador. Captura cerca de tres cuartas partes del margen disponible,
frente a menos de la mitad que captura elegir por pips.

**Lo que más pesa es poder construir, no producir.** Los coeficientes más grandes son
`trio_desarrollo`, `cuarteto_poblado` y `par_camino`: combinaciones completas de recursos. La
producción total entra después.

**El puerto alineado suma, aunque la correlación dijera lo contrario.** Al controlar por la
producción, el coeficiente se vuelve positivo. La correlación estaba confundida por la costa.

**Cuatro familias de estrategia salen solas de los datos** y se corresponden con jugadas
reconocibles: expansión, ciudades y desarrollo, puerto, y desequilibrada. La app usa esta
clasificación para explicar cada recomendación.

**Limitaciones.** El simulador juega en solitario, sin rivales que bloqueen ni intercambios entre
jugadores, y la política de construcción es fija. Los resultados valen para comparar parejas entre
sí, que es lo que la app necesita, no como pronóstico absoluto de una partida real.